A copy of notebooks [here](https://github.com/PacktPublishing/Deep-Reinforcement-Learning-Hands-On/blob/master/Chapter02) which I further play around with.

Also fixed a few bugs; e.g. line 13 needed one more `_` because `env.step(action)` returns a tuple with 5 objects.

In [27]:
import gymnasium as gym


if __name__ == "__main__":
    env = gym.make("CartPole-v0")

    total_reward = 0.0
    total_steps = 0
    obs = env.reset()

    while True:
        action = env.action_space.sample() # Random sample from action space; treat it as agent constnatly exploring.
        obs, reward, done, _, _ = env.step(action)
#         break
        total_reward += reward
        total_steps += 1
        if done:
            break

    print("Episode done in %d steps, total reward %.2f" % (total_steps, total_reward))

Episode done in 32 steps, total reward 32.00


In [28]:
action

np.int64(1)

In [29]:
help(env.step)

Help on method step in module gymnasium.wrappers.common:

step(action: 'ActType') -> 'tuple[ObsType, SupportsFloat, bool, bool, dict[str, Any]]' method of gymnasium.wrappers.common.TimeLimit instance
    Steps through the environment and if the number of steps elapsed exceeds ``max_episode_steps`` then truncate.
    
    Args:
        action: The environment step action
    
    Returns:
        The environment step ``(observation, reward, terminated, truncated, info)`` with `truncated=True`
        if the number of steps elapsed >= max episode steps



In [34]:
obs

array([ 0.04028323,  0.38175303, -0.21308918, -1.4525052 ], dtype=float32)

In [35]:
reward

1.0

In [36]:
done

True

In [38]:
env.reset

<bound method TimeLimit.reset of <TimeLimit<OrderEnforcing<PassiveEnvChecker<CartPoleEnv<CartPole-v0>>>>>>

In [42]:
env.observation_space

Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)

# Environment rendering

"Humanrendering" must open a popup window, which is problematic in Jupyter. Instead, I record steps as a video.

In [60]:
import gymnasium as gym
import moviepy

env = gym.make("CartPole-v1", render_mode="rgb_array")

env = gym.wrappers.RecordVideo(
    env,
    video_folder="videos",          # folder where video will be saved
#     episode_trigger=lambda e: True  # record every episode
)

obs, _ = env.reset()

while True:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, _ = env.step(action)

    if terminated or truncated:
        break

env.close()

In [59]:
help( gym.wrappers.RecordVideo)

Help on class RecordVideo in module gymnasium.wrappers.rendering:

class RecordVideo(gymnasium.core.Wrapper, typing.Generic, gymnasium.utils.record_constructor.RecordConstructorArgs)
 |  RecordVideo(env: 'gym.Env[ObsType, ActType]', video_folder: 'str', episode_trigger: 'Callable[[int], bool] | None' = None, step_trigger: 'Callable[[int], bool] | None' = None, video_length: 'int' = 0, name_prefix: 'str' = 'rl-video', fps: 'int | None' = None, disable_logger: 'bool' = True, gc_trigger: 'Callable[[int], bool] | None' = <function RecordVideo.<lambda> at 0x7f24e8dffce0>)
 |  
 |  Records videos of environment episodes using the environment's render function.
 |  
 |  .. py:currentmodule:: gymnasium.utils.save_video
 |  
 |  Usually, you only want to record episodes intermittently, say every hundredth episode or at every thousandth environment step.
 |  To do this, you can specify ``episode_trigger`` or ``step_trigger``.
 |  They should be functions returning a boolean that indicates whethe

# Cross entropy method on cartpole environment

In [54]:
import gymnasium as gym
import numpy as np
import moviepy
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from tensorboardX import SummaryWriter
from collections import namedtuple

HIDDEN_SIZE = 128
BATCH_SIZE = 16
PERCENTILE = 70

env = gym.make("CartPole-v0")
# env = gym.wrappers.Monitor(env, directory="mon", force=True)
obs_size = env.observation_space.shape[0]
n_actions = env.action_space.n

/opt/conda/lib/python3.11/site-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.deprecation(


In [10]:
class Net(nn.Module):
    def __init__(self, obs_size, hidden_size, n_actions):
        super(Net, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, n_actions)
        )

    def forward(self, x):
        return self.net(x)


In [14]:
net = Net(obs_size, HIDDEN_SIZE, n_actions)
objective = nn.CrossEntropyLoss()
optimizer = optim.Adam(params=net.parameters(), lr=0.01)
writer = SummaryWriter(comment="-cartpole")

So far same as original file. I wanted to look at several things.

In [15]:
env.action_space # 2 actions; move cartpole left or right.

Discrete(2)

In [17]:
env.observation_space # 4 observations (basically 'states'). Speed, angular speed, angle and relative position.

Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)

In [19]:
print(obs_size)
print(n_actions)

4
2


In [32]:
for name, param in net.named_parameters():
    print(name)
    print(param.shape)

net.0.weight
torch.Size([128, 4])
net.0.bias
torch.Size([128])
net.2.weight
torch.Size([2, 128])
net.2.bias
torch.Size([2])


In [22]:
Episode = namedtuple('Episode', field_names=['reward', 'steps'])
EpisodeStep = namedtuple('EpisodeStep', field_names=['observation', 'action'])

In [27]:
print(type(Episode), type(EpisodeStep), type(torch.nn.Module), type(str), type("hi"))
# From this, we see that `Episode` and `EpisodeStep` are actually classes themselves, which will be called to make
# instances of them.

<class 'type'> <class 'type'> <class 'type'> <class 'type'> <class 'str'>


## Training logic

This cross-entropy RL algorithm centers around `Net`, which accepts a set of observations (4 numbers indicating the state of the environment at the current timestep), and outputs logits which, when softmaxed, provide the probabilities of choosing each of the two possible actions. Effectively, `Net` is our policy $\pi(a|s)=\pi(a|\text{obs})$. 

The training logic is as follows:

- The `iterate_batch` function is responsible for creating batches of training data, in the form of 'episodes'. Each episode is a sequence of observations produced by the environment, actions chosen by the policy and the ensuing rewards and new observations. Within `iterate_batch`, `Net` is repeatedly provided environment observations and outputs action probabilities, which are used to choose which action to take. This repeats until the episode ends, at which point observations, actions and total reward are stored in an instance of the `Episode` class.

  `iterate_batch` produces an 'iterator' object when called; calling `next` on the iterator produces the next batch of episodes.

- Once `iterate_batch` produces a batch of training episodes, the next step in our algorithm is to throw away all but the "best" episodes. (The goodness of episodes is determined by their total reward.) This filtering for the best episodes is done by the `filter_batch` function.

- The filtered batches now provide the data used to train `Net`. The observations are inputs, and actions are the outputs which the network must learn to train. `torch.nn.CrossEntropyLoss` is used because there are two, mutually exclusive classes of actions (move cart left or right). **Thus, the main training loop consists of 1) generating good training data with `iterate_batches` and `filter_batch`, and 2) using the generated observation-action pairs to train the network `Net`.**

Additionally, `SummaryWriter` is used to keep logs. I haven't used the parent package `tensorboardX` before, so I need to investigate it a bit closer.

Below is the code implementing the training logic described above.

**NOTE: Bugfix.** In `iterate_batches`, the original line 5 was `obs = env.reset()`. Changed to `obs, _ = env.reset()` because we don't want the empty dictionary returned by `reset`.

In [38]:
def iterate_batches(env, net, batch_size):
    batch = []
    episode_reward = 0.0
    episode_steps = []
    obs = env.reset()
    sm = nn.Softmax(dim=1)
    while True:
        obs_v = torch.FloatTensor([obs])
        with torch.no_grad:
            act_probs_v = sm(net(obs_v))
        act_probs = act_probs_v.data.numpy()[0]
        action = np.random.choice(len(act_probs), p=act_probs)
        next_obs, reward, is_done, _ = env.step(action)
        episode_reward += reward
        episode_steps.append(EpisodeStep(observation=obs, action=action))
        if is_done:
            batch.append(Episode(reward=episode_reward, steps=episode_steps))
            episode_reward = 0.0
            episode_steps = []
            next_obs = env.reset()
            if len(batch) == batch_size:
                yield batch
                batch = []
        obs = next_obs


def filter_batch(batch, percentile):
    rewards = list(map(lambda s: s.reward, batch))
    reward_bound = np.percentile(rewards, percentile)
    reward_mean = float(np.mean(rewards))

    train_obs = []
    train_act = []
    for example in batch:
        if example.reward < reward_bound:
            continue
        train_obs.extend(map(lambda step: step.observation, example.steps))
        train_act.extend(map(lambda step: step.action, example.steps))

    train_obs_v = torch.FloatTensor(train_obs)
    train_act_v = torch.LongTensor(train_act)
    return train_obs_v, train_act_v, reward_bound, reward_mean


In [37]:
env = gym.make("CartPole-v0")
# env = gym.wrappers.Monitor(env, directory="mon", force=True)
obs_size = env.observation_space.shape[0]
n_actions = env.action_space.n

net = Net(obs_size, HIDDEN_SIZE, n_actions)
objective = nn.CrossEntropyLoss()
optimizer = optim.Adam(params=net.parameters(), lr=0.01)
writer = SummaryWriter(comment="-cartpole")

for iter_no, batch in enumerate(iterate_batches(env, net, BATCH_SIZE)):
    obs_v, acts_v, reward_b, reward_m = filter_batch(batch, PERCENTILE)
    optimizer.zero_grad()
    action_scores_v = net(obs_v)
    loss_v = objective(action_scores_v, acts_v)
    loss_v.backward()
    optimizer.step()
    print("%d: loss=%.3f, reward_mean=%.1f, reward_bound=%.1f" % (
        iter_no, loss_v.item(), reward_m, reward_b))
    writer.add_scalar("loss", loss_v.item(), iter_no)
    writer.add_scalar("reward_bound", reward_b, iter_no)
    writer.add_scalar("reward_mean", reward_m, iter_no)
    if reward_m > 199:
        print("Solved!")
        break
writer.close()

/tmp/ipykernel_11/134090989.py:8: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  obs_v = torch.FloatTensor([obs])


ValueError: expected sequence of length 4 at dim 2 (got 0)

In [63]:
def iterate_batches(env, net, batch_size):
    batch = []
    episode_reward = 0.0
    episode_steps = []
    obs, _ = env.reset()
    sm = nn.Softmax(dim=1)
    while True:
#         print("######") # Debuggign.
#         print(obs)
#         print(obs.shape)
        obs_v = torch.FloatTensor([obs])
#         print(obs_v.shape)
        with torch.no_grad():
            act_probs_v = sm(net(obs_v))
        act_probs = act_probs_v.data.numpy()[0]
        action = np.random.choice(len(act_probs), p=act_probs)
        next_obs, reward, is_done, _, _ = env.step(action)
        episode_reward += reward
        episode_steps.append(EpisodeStep(observation=obs, action=action))
        if is_done:
            batch.append(Episode(reward=episode_reward, steps=episode_steps))
            episode_reward = 0.0
            episode_steps = []
            next_obs, _ = env.reset()
            if len(batch) == batch_size:
                yield batch
                batch = []
        obs = next_obs

In [64]:
stuff = next(iterate_batches(env, net, BATCH_SIZE))

In [69]:
print(type(stuff))
print(len(stuff))
print(type(stuff[0]))
print()

<class 'list'>
16
<class '__main__.Episode'>


In [77]:
stuff[0].reward

30.0

In [76]:
stuff[0].steps # A list of steps.

[EpisodeStep(observation=array([ 0.03613358,  0.00030363,  0.00031798, -0.00172686], dtype=float32), action=1),
 EpisodeStep(observation=array([ 3.6139656e-02,  1.9542103e-01,  2.8343830e-04, -2.9430944e-01],
       dtype=float32), action=0),
 EpisodeStep(observation=array([ 0.04004807,  0.00029503, -0.00560275, -0.00153714], dtype=float32), action=0),
 EpisodeStep(observation=array([ 0.04005397, -0.19474612, -0.00563349,  0.2893728 ], dtype=float32), action=0),
 EpisodeStep(observation=array([ 3.6159053e-02, -3.8978729e-01,  1.5396242e-04,  5.8027369e-01],
       dtype=float32), action=1),
 EpisodeStep(observation=array([ 0.02836331, -0.19466749,  0.01175944,  0.28763923], dtype=float32), action=0),
 EpisodeStep(observation=array([ 0.02446996, -0.38995516,  0.01751222,  0.5840077 ], dtype=float32), action=0),
 EpisodeStep(observation=array([ 0.01667085, -0.58531797,  0.02919238,  0.88215536], dtype=float32), action=1),
 EpisodeStep(observation=array([ 0.00496449, -0.39060444,  0.04683

In [84]:
# Extracting observations and action for a given episode step.
print(stuff[0].steps[0].observation)
print(stuff[0].steps[0].action)

[ 0.03613358  0.00030363  0.00031798 -0.00172686]
1


We have fixed the errors in `iterate_batches`.

I also want to look more closely at how `filter_batch` behaves.

In [108]:
def filter_batch(batch, percentile):
    rewards = list(map(lambda s: s.reward, batch))
    print(f"Rewards: {rewards}.\n")
    reward_bound = np.percentile(rewards, percentile)
    reward_mean = float(np.mean(rewards))

    train_obs = []
    train_act = []
    counter = 0 # Counts total number of steps across all episodes with high reward.
    for example in batch:
        if example.reward < reward_bound:
            continue
        train_obs.extend(map(lambda step: step.observation, example.steps))
        train_act.extend(map(lambda step: step.action, example.steps))
        
        # My additional code for counting steps.
        counter += len(example.steps)
        print(f"Counter: {counter}.")

#     print(f"train_obs: {train_obs}.\n")
    train_obs_v = torch.FloatTensor(train_obs)
#     print(f"train_obs_v: {train_obs_v}.\n")
    train_act_v = torch.LongTensor(train_act)
#     print(f"train_act_v: {train_act_v}.\n")
    
    print(f"Example: {type(example)}.")
    print(type(train_obs), type(train_obs[0]))
    print(len(train_obs), train_obs[0].shape)
    print(train_obs_v.shape)
    print(train_act_v.shape)
    return train_obs_v, train_act_v, reward_bound, reward_mean

In [109]:
batch = stuff
train_obs_v, train_act_v, reward_bound, reward_mean = filter_batch(batch, percentile=70)

Rewards: [30.0, 18.0, 24.0, 52.0, 9.0, 14.0, 11.0, 26.0, 33.0, 43.0, 15.0, 27.0, 8.0, 16.0, 12.0, 14.0].

Counter: 30.
Counter: 82.
Counter: 115.
Counter: 158.
Counter: 185.
Example: <class '__main__.Episode'>.
<class 'list'> <class 'numpy.ndarray'>
185 (4,)
torch.Size([185, 4])
torch.Size([185])


We see that for each episode with reward greater than `reward_bound`, the observation-action pair are extracted for each step and appended to a bigger list. The `counter` confirms that steps are accumulated across the separate episodes into two big Tensors, `train_obs_v` and `train_act_v`. Also interesting is that `train_obs` was a list of arrays, and and turning it into a Tensor added an extra dimension (which is the 'batch' dimension for network training). It would be more robust against errors to specify which dimension to concatenate the arrays along, but the default behavior works out for us.

Let's try training again, with the corrected code.

**NOTE:** I've additionally wrapped the environment with `RecordVideo` to record the cartpole balancing task.

In [111]:
def iterate_batches(env, net, batch_size):
    batch = []
    episode_reward = 0.0
    episode_steps = []
    obs, _ = env.reset()
    sm = nn.Softmax(dim=1)
    while True:
#         print("######") # Debuggign.
#         print(obs)
#         print(obs.shape)
        obs_v = torch.FloatTensor([obs])
#         print(obs_v.shape)
        with torch.no_grad():
            act_probs_v = sm(net(obs_v))
        act_probs = act_probs_v.data.numpy()[0]
        action = np.random.choice(len(act_probs), p=act_probs)
        next_obs, reward, is_done, _, _ = env.step(action)
        episode_reward += reward
        episode_steps.append(EpisodeStep(observation=obs, action=action))
        if is_done:
            batch.append(Episode(reward=episode_reward, steps=episode_steps))
            episode_reward = 0.0
            episode_steps = []
            next_obs, _ = env.reset()
            if len(batch) == batch_size:
                yield batch
                batch = []
        obs = next_obs
        
def filter_batch(batch, percentile):
    rewards = list(map(lambda s: s.reward, batch))
    reward_bound = np.percentile(rewards, percentile)
    reward_mean = float(np.mean(rewards))

    train_obs = []
    train_act = []
    for example in batch:
        if example.reward < reward_bound:
            continue
        train_obs.extend(map(lambda step: step.observation, example.steps))
        train_act.extend(map(lambda step: step.action, example.steps))

    train_obs_v = torch.FloatTensor(train_obs)
    train_act_v = torch.LongTensor(train_act)
    return train_obs_v, train_act_v, reward_bound, reward_mean

###########################################################################################

env = gym.make("CartPole-v0", render_mode="rgb_array")
env = gym.wrappers.RecordVideo(
    env,
    video_folder="videos",          # folder where video will be saved
#     episode_trigger=lambda e: True  # record every episode
)
# env = gym.wrappers.Monitor(env, directory="mon", force=True)
obs_size = env.observation_space.shape[0]
n_actions = env.action_space.n

net = Net(obs_size, HIDDEN_SIZE, n_actions)
objective = nn.CrossEntropyLoss()
optimizer = optim.Adam(params=net.parameters(), lr=0.01)
writer = SummaryWriter(comment="-cartpole")

for iter_no, batch in enumerate(iterate_batches(env, net, BATCH_SIZE)):
    obs_v, acts_v, reward_b, reward_m = filter_batch(batch, PERCENTILE)
    optimizer.zero_grad()
    action_scores_v = net(obs_v)
    loss_v = objective(action_scores_v, acts_v)
    loss_v.backward()
    optimizer.step()
    print("%d: loss=%.3f, reward_mean=%.1f, reward_bound=%.1f" % (
        iter_no, loss_v.item(), reward_m, reward_b))
    writer.add_scalar("loss", loss_v.item(), iter_no)
    writer.add_scalar("reward_bound", reward_b, iter_no)
    writer.add_scalar("reward_mean", reward_m, iter_no)
    if reward_m > 199:
        print("Solved!")
        break
writer.close()

/opt/conda/lib/python3.11/site-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /workspace/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


0: loss=0.690, reward_mean=17.1, reward_bound=17.0
1: loss=0.673, reward_mean=18.4, reward_bound=18.0
2: loss=0.683, reward_mean=29.9, reward_bound=33.0
3: loss=0.673, reward_mean=21.8, reward_bound=24.0
4: loss=0.673, reward_mean=30.4, reward_bound=32.0
5: loss=0.652, reward_mean=27.2, reward_bound=30.5
6: loss=0.643, reward_mean=43.2, reward_bound=52.0
7: loss=0.627, reward_mean=39.5, reward_bound=45.5
8: loss=0.628, reward_mean=38.8, reward_bound=39.5
9: loss=0.605, reward_mean=43.2, reward_bound=40.5
10: loss=0.611, reward_mean=55.1, reward_bound=63.0
11: loss=0.593, reward_mean=46.4, reward_bound=57.5
12: loss=0.595, reward_mean=50.8, reward_bound=61.5
13: loss=0.582, reward_mean=45.5, reward_bound=51.0
14: loss=0.582, reward_mean=65.8, reward_bound=63.5
15: loss=0.566, reward_mean=70.2, reward_bound=73.0
16: loss=0.555, reward_mean=62.6, reward_bound=69.0
17: loss=0.569, reward_mean=57.1, reward_bound=63.0
18: loss=0.531, reward_mean=54.1, reward_bound=57.5
19: loss=0.537, reward

## Comments

- It does appear to work.
- `writer` created a folder called 'runs', with a bunch of files ending in '-cartpole'. These have no extensions, and I don't know how to access them. 
- 9 videos were recorded; their size on disk increased for each newer one that was made. This corresponds to the `Net`-policy doing a better job of keeping the pole balanced for a longer time. **I'm not entirely sure what prompts the recording of a video; there are 60 training runs, shouldn't a video be recorded for each one? TODO: Figure out.**

In [112]:
from IPython.display import Video

Video("videos/rl-video-episode-729.mp4") # Last video recorded. It might not correspond to the best version of `Net`; that
# depends on what causes a video to be recorded.